# Tramos

Cada fila de `data/processed/tramos.csv` es un tramo de calle del microcentro. `tipo_zona` sale de OpenStreetMap; `prioridad_recoleccion` es sintética y es la etiqueta.

Acá se agregan dos columnas binarias, se explora el dataset y se parte en train, validación y test. Sin entrenar modelo.

Raíz del repo y carga del CSV. Si ya existían `cerca_escuela_hospital` o `feria_o_evento`, se dropean antes de regenerarlas (idempotente).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data/processed/tramos.csv").is_file():
            return candidate
    raise FileNotFoundError("no está data/processed/tramos.csv; corré el notebook desde el repo")

ROOT = repo_root()
ZONA = "tipo_zona"
PRIORIDAD = "prioridad_recoleccion"
CERCA = "cerca_escuela_hospital"
FERIA = "feria_o_evento"
NUMERICAS = [
    "cantidad_contenedores",
    "dias_desde_ultima_recoleccion",
    "historico_quejas_vecinales",
]
BINARIAS = [CERCA, FERIA]
ZONAS = ["residencial", "comercial", "industrial"]
CLASES = ["alta", "media", "baja"]

tramos = pd.read_csv(ROOT / "data/processed/tramos.csv")
tramos = tramos.drop(columns=[c for c in BINARIAS if c in tramos.columns])
presentes = [zona for zona in ZONAS if (tramos[ZONA] == zona).any()]
tramos.shape

(1257, 11)

## Columnas sintéticas binarias

Mismo método que `backend/dataset/synthesize.py`: `numpy.random.Generator` y tasa por `tipo_zona`. Días y quejas son Poisson; estas dos son sí/no con Bernoulli.

Semilla 42 ya generó días, quejas y prioridad. Acá `SEMILLA = 43`. Primero `cerca_escuela_hospital`, después `feria_o_evento`. No se reescriben días, quejas ni prioridad.

In [2]:
SEMILLA = 43
PROBABILIDAD = {
    CERCA: {"residencial": 0.08, "comercial": 0.05, "industrial": 0.03},
    FERIA: {"residencial": 0.04, "comercial": 0.16, "industrial": 0.06},
}

def bernoulli_por_zona(zonas: pd.Series, probabilidad: dict[str, float], rng: np.random.Generator) -> np.ndarray:
    tasa = zonas.map(probabilidad).to_numpy(dtype=float)
    return (rng.random(len(zonas)) < tasa).astype(int)

def agregar(frame: pd.DataFrame, semilla: int) -> pd.DataFrame:
    rng = np.random.default_rng(semilla)
    out = frame.copy()
    zonas = out[ZONA].astype(str)
    out[CERCA] = bernoulli_por_zona(zonas, PROBABILIDAD[CERCA], rng)
    out[FERIA] = bernoulli_por_zona(zonas, PROBABILIDAD[FERIA], rng)
    return out

primera = agregar(tramos, SEMILLA)
segunda = agregar(tramos, SEMILLA)
if not primera[[CERCA, FERIA]].equals(segunda[[CERCA, FERIA]]):
    raise RuntimeError("la misma semilla no repitió las columnas")

tramos = primera
tramos[[CERCA, FERIA]].mean().round(3)

cerca_escuela_hospital    0.076
feria_o_evento            0.077
dtype: float64

## Una fila

Columnas del dataset mínimo, calle y las dos binarias nuevas.

In [3]:
tramos[["tramo_id", "nombre_calle", ZONA, *NUMERICAS, *BINARIAS, PRIORIDAD]].head(8)

,tramo_id,nombre_calle,tipo_zona,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales,cerca_escuela_hospital,feria_o_evento,prioridad_recoleccion
0,1,Teniente General Juan Domingo Perón,residencial,0,4,2,0,0,media
1,2,Avenida 9 de Julio,residencial,0,4,1,1,0,media
2,3,Avenida 9 de Julio,residencial,0,5,1,1,0,media
3,4,Teniente General Juan Domingo Perón,residencial,0,1,0,0,0,baja
4,5,Viamonte,residencial,0,7,0,0,0,alta
5,6,Avenida 9 de Julio,residencial,0,1,1,0,0,baja
6,7,Avenida 9 de Julio,residencial,0,4,1,0,0,media
7,8,Viamonte,residencial,0,2,1,0,0,baja


## Tipo de zona

Industrial figura en el conteo aunque no haya tramos.

In [4]:
zonas = tramos[ZONA].value_counts().reindex(ZONAS, fill_value=0)
pd.DataFrame({"tramos": zonas, "proporcion": (zonas / len(tramos)).round(3)})

,tramos,proporcion
tipo_zona,,
residencial,859,0.683
comercial,398,0.317
industrial,0,0.000


## Cerca de escuela u hospital

In [5]:
def conteo_binaria(columna: str) -> pd.DataFrame:
    cantidades = tramos[columna].value_counts().reindex([0, 1], fill_value=0)
    return pd.DataFrame(
        {"tramos": cantidades, "proporcion": (cantidades / len(tramos)).round(3)}
    )

conteo_binaria(CERCA)

,tramos,proporcion
cerca_escuela_hospital,,
0,1162,0.924
1,95,0.076


## Feria o evento

In [6]:
conteo_binaria(FERIA)

,tramos,proporcion
feria_o_evento,,
0,1160,0.923
1,97,0.077


Proporción de sí (1) dentro de cada tipo de zona.

In [7]:
tramos.groupby(ZONA)[BINARIAS].mean().reindex(ZONAS).round(3)

,cerca_escuela_hospital,feria_o_evento
tipo_zona,,
residencial,0.078,0.043
comercial,0.070,0.151
industrial,NaN,NaN


## Prioridad global

Etiqueta cerca de alta 20 %, media 35 %, baja 45 %.

In [8]:
prioridad = tramos[PRIORIDAD].value_counts().reindex(CLASES)
pd.DataFrame({"tramos": prioridad, "proporcion": (prioridad / len(tramos)).round(3)})

,tramos,proporcion
prioridad_recoleccion,,
alta,251,0.20
media,440,0.35
baja,566,0.45


Promedio de features numéricas y binarias por tipo de zona (solo zonas presentes).

In [9]:
tramos.groupby(ZONA)[NUMERICAS + BINARIAS].mean().reindex(presentes).round(2)

,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales,cerca_escuela_hospital,feria_o_evento
tipo_zona,,,,,
residencial,0.43,3.20,1.09,0.08,0.04
comercial,1.31,2.18,1.88,0.07,0.15


## Guardar

Mismo `data/processed/tramos.csv`, con las dos columnas nuevas. El resto sin cambios.

In [10]:
destino = ROOT / "data/processed/tramos.csv"
tramos.to_csv(destino, index=False)
destino.name, tramos.shape

('tramos.csv', (1257, 13))

## Tres conjuntos: train, validación y test

- **Train (60 %):** ajusta parámetros cuando entrenemos.
- **Validación (20 %):** elige modelos o hiperparámetros.
- **Test (20 %):** estimación final; se mira una sola vez y no se toca.

La cátedra suele usar solo train/test y la validación cruzada vive dentro del train. En este grupo pidieron un 20 % de validación aparte.

**Stratify:** `prioridad_recoleccion` no está balanceada (alta ~20 %, media ~35 %, baja ~45 %). `stratify=prioridad_recoleccion` reparte cada clase con la misma proporción en cada corte.

**random_state=42:** misma semilla que `SEED` en `backend/config.py` al generar el dataset.

1. `train_test_split` con `test_size=0.40` y stratify: 60 % train, 40 % resto.
2. Sobre ese 40 %, otro split con `test_size=0.50` y stratify: 20 % validación y 20 % test del total.

In [11]:
from sklearn.model_selection import train_test_split

y = tramos[PRIORIDAD]

train_idx, rest_idx = train_test_split(
    tramos.index,
    test_size=0.40,
    random_state=42,
    stratify=y,
)
validacion_idx, test_idx = train_test_split(
    rest_idx,
    test_size=0.50,
    random_state=42,
    stratify=y.loc[rest_idx],
)

train = tramos.loc[train_idx].copy()
validacion = tramos.loc[validacion_idx].copy()
test = tramos.loc[test_idx].copy()

len(train), len(validacion), len(test)

(754, 251, 252)

Filas y proporción de cada conjunto (~0.60 / 0.20 / 0.20).

In [12]:
n = len(tramos)
partes = pd.Series(
    {"train": len(train), "validacion": len(validacion), "test": len(test)},
    name="filas",
)
pd.DataFrame({"filas": partes, "proporcion": (partes / n).round(3)})

,filas,proporcion
train,754,0.6
validacion,251,0.2
test,252,0.2


Proporción de alta, media y baja dentro de train, validación y test.

In [13]:
def prop_prioridad(df: pd.DataFrame) -> pd.Series:
    return df[PRIORIDAD].value_counts(normalize=True).reindex(CLASES).round(3)

pd.DataFrame(
    {
        "train": prop_prioridad(train),
        "validacion": prop_prioridad(validacion),
        "test": prop_prioridad(test),
    }
)

,train,validacion,test
prioridad_recoleccion,,,
alta,0.20,0.199,0.198
media,0.35,0.351,0.349
baja,0.45,0.450,0.452


Cada `tramo_id` en un solo conjunto.

In [14]:
ids_train = set(train["tramo_id"])
ids_validacion = set(validacion["tramo_id"])
ids_test = set(test["tramo_id"])

assert ids_train.isdisjoint(ids_validacion)
assert ids_train.isdisjoint(ids_test)
assert ids_validacion.isdisjoint(ids_test)
assert len(ids_train) + len(ids_validacion) + len(ids_test) == len(tramos)

"train, validacion y test no comparten tramo_id"

'train, validacion y test no comparten tramo_id'